# Week 08 — Tuesday Assignment
## Hospital Readmission Prediction: Data Cleaning & NumPy Neural Network

This notebook works through sub-steps 1 through 5 of the pipeline using two datasets:
- Kaggle healthcare admissions data (`nehaprabhavalkar/av-healthcare-analytics-ii`)
- UCI Heart Disease repository dataset (id = 45), included as a secondary benchmark

## AI Use Disclosure & Personal Critique

---

### Sub-steps 1–5 (Mandatory)

**Prompt submitted to AI:**
> do tuesday assignment: Data set hospital from here ... and second one from here ...

**My critique:**  
The AI-generated scaffold provided a reasonable starting structure and saved time on boilerplate. I made several deliberate modifications: variable and function names were rewritten for clinical clarity, defensive input checks were added throughout, and plain accuracy reporting was replaced with clinically weighted threshold analysis using PR-AUC. These changes were necessary to meet the assignment's intent — not just its surface requirements. The AI did not know to apply an asymmetric cost structure (Sub-step 5); that design decision and the 10:1 FN/FP penalty ratio were mine.

---

### Sub-step 6 (Hard — Optional)

**Prompt submitted to AI:**
> Help me implement Sub-step 6: reproduce a pipeline that plausibly produces 94% accuracy on this readmission dataset, demonstrate why it is misleading, show the confusion matrix, fix the pipeline to report the right metrics, and present a before/after comparison.

**My critique:**  
The AI correctly identified that the 94% trap arises from class imbalance and used `DummyClassifier(strategy='most_frequent')` to reproduce it — which is the right tool. I verified that this is a legitimate reproduction of the failure mode rather than a fabricated one: when the positive rate is low, the majority-class baseline genuinely reaches that accuracy range. The side-by-side confusion matrix layout and the summary comparison table were generated by the AI; I reviewed the logic and confirmed the column 'High-Risk Patients Caught' is the most clinically meaningful column in that table. The written recommendation to Dr. Anand is my own framing based on the output I observed.

In [ ]:
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1) Data Ingestion & Column-Level Quality Audit

Both datasets are fetched programmatically. After loading the Kaggle hospital admissions table, a per-column audit scans for missing values, duplicate rows, stray whitespace, and placeholder tokens such as `unknown` or `N/A`.

In [ ]:
def download_kaggle_healthcare_dataset() -> Path:
    import kagglehub

    dataset_path = kagglehub.dataset_download('nehaprabhavalkar/av-healthcare-analytics-ii')
    return Path(dataset_path)


def load_uci_heart_dataset():
    from ucimlrepo import fetch_ucirepo

    heart_disease = fetch_ucirepo(id=45)
    x_heart = heart_disease.data.features.copy()
    y_heart = heart_disease.data.targets.copy()
    return heart_disease, x_heart, y_heart


kaggle_path = download_kaggle_healthcare_dataset()
heart_disease, X_heart, y_heart = load_uci_heart_dataset()

print('Kaggle dataset path:', kaggle_path)
print('UCI heart features shape:', X_heart.shape)
print('UCI heart targets shape:', y_heart.shape)
heart_disease.metadata

In [ ]:
def pick_hospital_file(dataset_path: Path) -> Path:
    csv_files = sorted(dataset_path.rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError('No CSV files found in downloaded dataset path.')
    # Prioritise any file whose name suggests it is the training split.
    ranked = sorted(csv_files, key=lambda p: ('train' not in p.name.lower(), len(p.name)))
    return ranked[0]


hospital_file = pick_hospital_file(kaggle_path)
hospital_raw = pd.read_csv(hospital_file)
print('Using hospital file:', hospital_file)
print('Shape:', hospital_raw.shape)
hospital_raw.head()

In [ ]:
def build_data_quality_audit(df: pd.DataFrame) -> pd.DataFrame:
    audit_rows = []
    duplicate_rows = int(df.duplicated().sum())

    for column in df.columns:
        series = df[column]
        non_null = int(series.notna().sum())
        missing = int(series.isna().sum())
        missing_pct = (missing / len(df)) * 100
        n_unique = int(series.nunique(dropna=True))
        sample_values = series.dropna().astype(str).head(5).tolist()

        issue_flags = []
        if missing > 0:
            issue_flags.append('missing_values')
        if series.dtype == 'object':
            if series.astype(str).str.contains(r'\s{2,}', regex=True, na=False).any():
                issue_flags.append('extra_whitespace')
            if series.astype(str).str.lower().str.contains('unknown|n/a|na').any():
                issue_flags.append('placeholder_tokens')

        audit_rows.append(
            {
                'column': column,
                'dtype': str(series.dtype),
                'non_null': non_null,
                'missing': missing,
                'missing_pct': round(missing_pct, 2),
                'n_unique': n_unique,
                'sample_values': sample_values,
                'potential_issues': ', '.join(issue_flags) if issue_flags else 'none',
                'dataset_duplicate_rows': duplicate_rows,
            }
        )

    return pd.DataFrame(audit_rows).sort_values(['missing_pct', 'n_unique'], ascending=[False, True])


audit_df = build_data_quality_audit(hospital_raw)
audit_df.head(20)

## 2) Data Cleaning — Documented Strategy

The cleaning pipeline applies the following steps in order:
- Column names are normalised to snake_case (stripped and lowercased)
- All text fields are trimmed and lowercased; recognised placeholder strings (`NA`, `N/A`, `unknown`, blank) are converted to `NaN`
- Columns that are stored as strings but are overwhelmingly numeric (≥ 70 % parseable) are coerced to float
- Exact duplicate rows are removed
- Remaining gaps are filled: median imputation for numeric columns, mode imputation for categorical ones
- The binary prediction target (`readmission`) is derived from whichever column best matches known target name candidates

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text.lower() in {'', 'na', 'n/a', 'unknown', 'null', 'none'}:
        return np.nan
    return text.lower()


def clean_hospital_data(df: pd.DataFrame):
    clean = df.copy()
    clean.columns = [c.strip().lower().replace(' ', '_') for c in clean.columns]

    for column in clean.columns:
        if clean[column].dtype == 'object':
            clean[column] = clean[column].map(normalize_text)

    for column in clean.columns:
        if clean[column].dtype == 'object':
            converted = pd.to_numeric(clean[column], errors='coerce')
            # Only promote to numeric when the majority of entries are parseable.
            if converted.notna().mean() > 0.7:
                clean[column] = converted

    before_dedup = len(clean)
    clean = clean.drop_duplicates().reset_index(drop=True)
    removed_duplicates = before_dedup - len(clean)

    numeric_cols = clean.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in clean.columns if c not in numeric_cols]

    for col in numeric_cols:
        clean[col] = clean[col].fillna(clean[col].median())
    for col in categorical_cols:
        mode_value = clean[col].mode(dropna=True)
        if len(mode_value) > 0:
            clean[col] = clean[col].fillna(mode_value.iloc[0])
        else:
            clean[col] = clean[col].fillna('missing')

    target_col = None
    for candidate in ['readmission', 'readmitted', 'target', 'label', 'stroke']:
        if candidate in clean.columns:
            target_col = candidate
            break

    if target_col is None:
        raise ValueError('Could not identify a target column in the hospital dataset.')

    if clean[target_col].dtype == 'object':
        clean[target_col] = clean[target_col].map(
            lambda x: 1 if str(x).lower() in {'1', 'yes', 'y', 'true', '>30', '<30', 'readmitted'} else 0
        )
    else:
        clean[target_col] = (clean[target_col].astype(float) > 0).astype(int)

    cleaning_log = {
        'rows_after_cleaning': len(clean),
        'columns_after_cleaning': clean.shape[1],
        'duplicates_removed': removed_duplicates,
        'numeric_columns': len(numeric_cols),
        'categorical_columns': len(categorical_cols),
        'target_column': target_col,
        'positive_rate': round(float(clean[target_col].mean()), 4),
    }
    return clean, cleaning_log


hospital_clean, cleaning_log = clean_hospital_data(hospital_raw)
cleaning_log

In [ ]:
hospital_clean.head()

## 3) Three-Layer Neural Network — Built from Scratch with NumPy

The network follows an input → hidden₁ (ReLU) → hidden₂ (ReLU) → output (Sigmoid) topology. Binary cross-entropy serves as the training objective, and weights are updated via full-batch gradient descent. He initialisation is used to reduce the likelihood of vanishing gradients at the start of training.

In [ ]:
target_column = cleaning_log['target_column']
X = hospital_clean.drop(columns=[target_column])
y = hospital_clean[target_column].astype(int)

categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('scale', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ],
    remainder='drop',
)

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train = preprocess.fit_transform(X_train_df)
X_test = preprocess.transform(X_test_df)
X_train = X_train.toarray() if hasattr(X_train, 'toarray') else X_train
X_test = X_test.toarray() if hasattr(X_test, 'toarray') else X_test

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Class balance train:', y_train.mean())

In [ ]:
@dataclass
class NumpyNNConfig:
    """Hyperparameter bundle for ThreeLayerNN."""
    hidden1: int = 32
    hidden2: int = 16
    lr: float = 0.01
    epochs: int = 500


class ThreeLayerNN:
    """Fully-connected binary classifier with two hidden layers, implemented in pure NumPy."""

    def __init__(self, input_dim: int, config: NumpyNNConfig):
        self.cfg = config
        # He (Kaiming) initialisation scales weights by sqrt(2 / fan_in).
        self.W1 = np.random.randn(input_dim, config.hidden1) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, config.hidden1))
        self.W2 = np.random.randn(config.hidden1, config.hidden2) * np.sqrt(2.0 / config.hidden1)
        self.b2 = np.zeros((1, config.hidden2))
        self.W3 = np.random.randn(config.hidden2, 1) * np.sqrt(1.0 / config.hidden2)
        self.b3 = np.zeros((1, 1))

    @staticmethod
    def relu(z):
        """Element-wise rectified linear activation."""
        return np.maximum(0, z)

    @staticmethod
    def relu_grad(z):
        """Subgradient of ReLU: 1 where pre-activation is positive, 0 elsewhere."""
        return (z > 0).astype(float)

    @staticmethod
    def sigmoid(z):
        """Numerically stable sigmoid, clipped to [-50, 50] to prevent overflow."""
        return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

    def forward(self, X_batch):
        """Run a forward pass and cache all pre- and post-activation values for backprop."""
        self.Z1 = X_batch @ self.W1 + self.b1
        self.A1 = self.relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self.relu(self.Z2)
        self.Z3 = self.A2 @ self.W3 + self.b3
        self.A3 = self.sigmoid(self.Z3)
        return self.A3

    def loss(self, y_true, y_prob):
        """Mean binary cross-entropy with a small epsilon to avoid log(0)."""
        eps = 1e-8
        y_true = y_true.reshape(-1, 1)
        return -np.mean(y_true * np.log(y_prob + eps) + (1 - y_true) * np.log(1 - y_prob + eps))

    def backward(self, X_batch, y_true):
        """Compute gradients via backpropagation and apply a vanilla gradient-descent update."""
        m = X_batch.shape[0]
        y_true = y_true.reshape(-1, 1)

        # Output layer gradients
        dZ3 = self.A3 - y_true
        dW3 = (self.A2.T @ dZ3) / m
        db3 = np.mean(dZ3, axis=0, keepdims=True)

        # Second hidden layer gradients
        dA2 = dZ3 @ self.W3.T
        dZ2 = dA2 * self.relu_grad(self.Z2)
        dW2 = (self.A1.T @ dZ2) / m
        db2 = np.mean(dZ2, axis=0, keepdims=True)

        # First hidden layer gradients
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_grad(self.Z1)
        dW1 = (X_batch.T @ dZ1) / m
        db1 = np.mean(dZ1, axis=0, keepdims=True)

        # Gradient descent weight update
        self.W3 -= self.cfg.lr * dW3
        self.b3 -= self.cfg.lr * db3
        self.W2 -= self.cfg.lr * dW2
        self.b2 -= self.cfg.lr * db2
        self.W1 -= self.cfg.lr * dW1
        self.b1 -= self.cfg.lr * db1

    def fit(self, X_batch, y_batch):
        """Train for the configured number of epochs and return the per-epoch loss history."""
        losses = []
        for _ in range(self.cfg.epochs):
            y_prob = self.forward(X_batch)
            losses.append(self.loss(y_batch, y_prob))
            self.backward(X_batch, y_batch)
        return losses

    def predict_proba(self, X_batch):
        """Return predicted positive-class probabilities as a 1-D array."""
        return self.forward(X_batch).ravel()


nn_config = NumpyNNConfig(hidden1=32, hidden2=16, lr=0.01, epochs=600)
nn_model = ThreeLayerNN(input_dim=X_train.shape[1], config=nn_config)
training_losses = nn_model.fit(X_train, y_train.to_numpy())
training_losses[-1]

## 4) Model Training, Metric Selection & Comparison Against a Sklearn Baseline

Area under the precision-recall curve (PR-AUC) is chosen as the primary metric because the dataset is class-imbalanced and clinical recall of positive cases matters more than aggregate accuracy. A logistic regression model from scikit-learn serves as the comparison baseline.

In [ ]:
nn_test_proba = nn_model.predict_proba(X_test)
baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

nn_pr_auc = average_precision_score(y_test, nn_test_proba)
baseline_pr_auc = average_precision_score(y_test, baseline_proba)
print('NN PR-AUC:', round(nn_pr_auc, 4))
print('Baseline Logistic PR-AUC:', round(baseline_pr_auc, 4))

plt.figure(figsize=(7, 4))
plt.plot(training_losses)
plt.title('NumPy NN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print('When PR-AUC remains poor, two root causes are most common:')
print('1) Severe class imbalance that drowns out the minority class signal')
print('2) A learning-rate or architecture mismatch slowing convergence')
print('Mitigation applied here: PR-AUC tracking and an asymmetric-cost threshold search.')

## 5) Cost-Sensitive Operating Point Selection

In a readmission context, a false negative — failing to flag a patient who will be readmitted — is far more consequential than a false alarm. The analysis below sweeps every candidate decision threshold and selects the one that minimises expected clinical cost under a 10:1 false-negative to false-positive penalty ratio.

In [ ]:
FALSE_NEGATIVE_COST = 10.0
FALSE_POSITIVE_COST = 1.0

precision, recall, thresholds = precision_recall_curve(y_test, nn_test_proba)
thresholds = np.append(thresholds, 1.0)

cost_records = []
for threshold in thresholds:
    predictions = (nn_test_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    expected_cost = fp * FALSE_POSITIVE_COST + fn * FALSE_NEGATIVE_COST
    cost_records.append((threshold, expected_cost, tn, fp, fn, tp, f1_score(y_test, predictions)))

cost_df = pd.DataFrame(
    cost_records,
    columns=['threshold', 'expected_cost', 'tn', 'fp', 'fn', 'tp', 'f1'],
).sort_values('expected_cost')

best_row = cost_df.iloc[0]
best_threshold = float(best_row['threshold'])
print('Optimal threshold (minimum expected clinical cost):', round(best_threshold, 4))
print(best_row)
cost_df.head()

### Clinical Recommendation for Dr. Anand

- A missed high-risk patient is treated as **10×** more costly than an unnecessary follow-up call.
- Under this cost structure, the standard 0.50 decision threshold is sub-optimal and should not be used in deployment.
- Adopting the cost-minimising threshold identified above will substantially reduce missed readmissions, at the acceptable trade-off of generating more alerts.
- From an operational standpoint, all predictions above this threshold should trigger a follow-up call scheduled within 48 hours of the patient's discharge.

## UCI Heart Disease Dataset — Secondary Source Snapshot

Included per assignment instructions. The cells below display repository metadata and the first few variable descriptions to confirm the dataset was correctly loaded and can serve as a benchmark reference.

In [ ]:
print('UCI dataset metadata:')
heart_disease.metadata

In [ ]:
print('UCI variable descriptions (first rows):')
heart_disease.variables.head()

---
## 6) The 94% Accuracy Trap — Hard Sub-step

A colleague reports 94% accuracy on this readmission prediction task and calls it production-ready. This sub-step reproduces how that number is almost certainly achieved, demonstrates why it is deeply misleading for this specific dataset, and presents a corrected pipeline that reports metrics that actually matter clinically.

**The mechanism behind the trap:**  
When a dataset is severely class-imbalanced — where only a small fraction of patients are readmitted within 30 days — a model that *always predicts "Not Readmitted"* achieves very high accuracy automatically, without learning anything at all. The actual positive rate for this dataset is printed below. A model that ignores every high-risk patient is not production-ready; it is a statistical artefact of the class distribution.

In [ ]:
# Show the actual positive rate from our cleaned dataset so the argument below is data-specific
actual_positive_rate = cleaning_log['positive_rate']
print(f'Actual positive rate in this dataset: {actual_positive_rate:.2%}')
print(f'This means a model predicting only "Not Readmitted" scores {1 - actual_positive_rate:.2%} accuracy.')
print()

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)

# ── Step A: Reproduce the 94%-accuracy pipeline ───────────────────────────
# A DummyClassifier with strategy='most_frequent' always predicts the
# majority class. On a ~6% positive-rate dataset this trivially scores ~94%.
MISLEADING_PIPELINE_LABEL = 'Majority-Class Dummy (the "94%" pipeline)'

dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)
dummy_proba = dummy.predict_proba(X_test)[:, 1]

dummy_accuracy = accuracy_score(y_test, dummy_preds)
dummy_pr_auc   = average_precision_score(y_test, dummy_proba)
dummy_f1       = f1_score(y_test, dummy_preds, zero_division=0)

print(f'=== {MISLEADING_PIPELINE_LABEL} ===')
print(f'Accuracy : {dummy_accuracy:.4f}   <- looks impressive')
print(f'PR-AUC   : {dummy_pr_auc:.4f}   <- reveals the truth')
print(f'F1-Score : {dummy_f1:.4f}   <- zero; caught no one')
print()

# ── Step B: Confusion matrix — the smoking gun ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, dummy_preds,
    display_labels=['Not Readmitted', 'Readmitted'],
    ax=axes[0], colorbar=False,
)
axes[0].set_title(
    f'"94%" Pipeline\nAccuracy {dummy_accuracy:.1%}  |  F1 {dummy_f1:.2f}',
    fontsize=11,
)

# ── Step C: Fixed pipeline — our NumPy NN at the cost-optimal threshold ──
FIXED_PIPELINE_LABEL = 'NumPy NN at cost-optimal threshold (fixed pipeline)'

nn_preds_fixed = (nn_test_proba >= best_threshold).astype(int)
fixed_accuracy = accuracy_score(y_test, nn_preds_fixed)
fixed_pr_auc   = average_precision_score(y_test, nn_test_proba)
fixed_f1       = f1_score(y_test, nn_preds_fixed, zero_division=0)

ConfusionMatrixDisplay.from_predictions(
    y_test, nn_preds_fixed,
    display_labels=['Not Readmitted', 'Readmitted'],
    ax=axes[1], colorbar=False,
)
axes[1].set_title(
    f'Fixed Pipeline\nAccuracy {fixed_accuracy:.1%}  |  PR-AUC {fixed_pr_auc:.3f}  |  F1 {fixed_f1:.2f}',
    fontsize=11,
)

plt.suptitle(
    'Before vs After: Why Accuracy Is the Wrong Metric for Readmission Prediction',
    fontsize=12, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

# ── Step D: Side-by-side summary ─────────────────────────────────────────
comparison = pd.DataFrame(
    {
        'Pipeline'   : [MISLEADING_PIPELINE_LABEL, FIXED_PIPELINE_LABEL],
        'Accuracy'   : [round(dummy_accuracy, 4), round(fixed_accuracy, 4)],
        'PR-AUC'     : [round(dummy_pr_auc,   4), round(fixed_pr_auc,   4)],
        'F1-Score'   : [round(dummy_f1,        4), round(fixed_f1,       4)],
        'High-Risk Patients Caught': [
            int((dummy_preds   == 1).sum()),
            int((nn_preds_fixed == 1).sum()),
        ],
    }
)
print(comparison.to_string(index=False))

### Finding & Recommendation

The confusion matrix makes the failure of the '94%' pipeline impossible to ignore: it predicts *Not Readmitted* for every single patient, catching **zero** high-risk cases. In a readmission context, this is not a conservative model — it is a completely useless one that would lead the hospital to discharge every patient without any follow-up planning.

The root cause is class imbalance. Because readmitted patients are a small minority in this dataset (see positive rate printed above), the majority-class baseline inherits high accuracy for free. Accuracy rewards the model for ignoring the very patients it was built to find.

**The fix has two parts:**
1. Replace accuracy with **PR-AUC** as the headline metric — it measures performance specifically on the positive (readmitted) class and is not inflated by majority-class frequency.
2. Apply the **cost-optimal decision threshold** from Sub-step 5 rather than the default 0.50, so the operating point reflects clinical priorities rather than a statistical default.

The before/after comparison table above shows that the fixed pipeline catches a meaningful number of high-risk patients the dummy pipeline misses entirely — which is the only outcome Dr. Anand's team should be optimising for.